# 02 - Black-Scholes Greeks and Scenario Analysis

**Objective:** Use the cleaned option surface to connect implied volatility with option risk sensitivities.

**Inputs:** `data/processed/AAPL_surface_checked.csv` from notebook 01, with fallback to the frozen sample under `data/sample/`.

**Outputs:** Greeks by option row and scenario PnL figures under `reports/figures/`.

**Key takeaway:** Black-Scholes provides the project baseline for pricing, IV inversion, Greeks, and sanity checks before moving to stochastic volatility.




In [ ]:
from pathlib import Path
import os
import sys
import tempfile

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "heston_matplotlib_cache"))

FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.black_scholes import greeks, price

In [ ]:
TICKER = "AAPL"
checked_path = PROJECT_ROOT / "data" / "processed" / f"{TICKER}_surface_checked.csv"
sample_checked_path = PROJECT_ROOT / "data" / "sample" / f"{TICKER}_canonical_surface_checked.csv"

if checked_path.exists():
    load_path = checked_path
elif sample_checked_path.exists():
    load_path = sample_checked_path
else:
    raise FileNotFoundError("Run notebook 01 first or keep the canonical sample CSV under data/sample/.")

df = pd.read_csv(load_path, parse_dates=["expiry"])
spot = float((df["strike"] / df["moneyness"]).median())
print(f"Loaded {load_path.relative_to(PROJECT_ROOT)}")
print(f"Rows: {len(df):,}; spot={spot:.4f}")



## Greeks

For each contract we compute delta, gamma, vega, theta, and rho using the Black-Scholes model with each row's quoted implied volatility and inferred expiry-level rate/dividend inputs.

In [ ]:
def row_greeks(row):
    g = greeks(
        spot=spot,
        strike=float(row["strike"]),
        maturity=float(row["T"]),
        rate=float(row.get("r", 0.0)),
        vol=float(row["iv_clamped"]),
        side=row["side"],
        dividend=float(row.get("q", 0.0)),
    )
    return pd.Series(g.__dict__)

df_greeks = df.copy()
df_greeks[["delta", "gamma", "vega", "theta", "rho"]] = df_greeks.apply(row_greeks, axis=1)
df_greeks.head()

In [ ]:
for expiry, sub in df_greeks.groupby("expiry"):
    sub = sub.sort_values("strike")
    plt.figure(figsize=(7, 4))
    for side, marker in [("C", "o"), ("P", "x")]:
        tmp = sub[sub["side"] == side]
        if len(tmp):
            plt.plot(tmp["strike"], tmp["delta"], marker=marker, label=side)
    plt.axhline(0.0, linestyle="--", linewidth=0.8)
    plt.title(f"Delta by strike - {pd.to_datetime(expiry).date()}")
    plt.xlabel("Strike")
    plt.ylabel("Delta")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"delta_{pd.to_datetime(expiry).date()}.png", dpi=220)
    plt.close()

print(f"Saved delta charts to {FIG_DIR.relative_to(PROJECT_ROOT)}")

## Scenario Repricing

A scenario engine shocks spot, implied volatility, and calendar time, then reprices each contract with Black-Scholes. This is not a trading recommendation; it is a compact way to visualize risk exposure.

In [ ]:
def scenario_reprice(df, spot_shift_pct=0.0, vol_shift_abs=0.0, days_forward=0):
    out = df.copy()
    shocked_spot = spot * (1.0 + spot_shift_pct)
    out["T_shocked"] = np.maximum(out["T"].astype(float) - days_forward / 365.25, 0.0)
    out["iv_shocked"] = (out["iv_clamped"].astype(float) + vol_shift_abs).clip(lower=0.0001)
    out["price_new"] = [
        price(
            shocked_spot,
            row.strike,
            row.T_shocked,
            getattr(row, "r", 0.0),
            row.iv_shocked,
            row.side,
            getattr(row, "q", 0.0),
        )
        for row in out.itertuples()
    ]
    out["pnl"] = out["price_new"] - out["target_price"]
    return out

scenario = scenario_reprice(df_greeks, spot_shift_pct=0.02, vol_shift_abs=0.01, days_forward=0)
scenario[["expiry", "side", "strike", "target_price", "price_new", "pnl"]].head()

In [ ]:
focus_expiry = sorted(scenario["expiry"].unique())[0]
sub = scenario[scenario["expiry"] == focus_expiry].sort_values("strike")
colors = np.where(sub["pnl"] >= 0, "tab:green", "tab:red")

plt.figure(figsize=(8, 4.5))
plt.bar(sub["strike"], sub["pnl"], color=colors)
plt.axhline(0.0, linestyle="--", linewidth=0.8)
plt.title(f"Scenario PnL - {pd.to_datetime(focus_expiry).date()} - spot +2%, vol +1pt")
plt.xlabel("Strike")
plt.ylabel("PnL vs target price")
plt.tight_layout()
plt.savefig(FIG_DIR / f"scenario_pnl_{pd.to_datetime(focus_expiry).date()}.png", dpi=220)
plt.close()

## Takeaway

The Greek and scenario diagnostics establish the deterministic-volatility benchmark that the Heston model extends in the next notebook.

